In [1]:
%pip install --upgrade torchao

In [2]:
%pip install "pyarrow>=19.0.0,<24"

In [3]:
import torch

is_cuda_available = torch.cuda.is_available()
print(f"CUDA (GPU) Available: {is_cuda_available}")
if is_cuda_available:
    print(f"Current GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("You are currently running on a CPU. Go to 'Runtime' -> 'Change runtime type' -> select 'T4 GPU' to resolve the warnings and speed up execution!")

CUDA (GPU) Available: True
Current GPU Device: Tesla T4


In [4]:
%pip install --upgrade transformers peft datasets "pyarrow>=19.0.0,<24"
!rm -rf /content/=0.10.0

import os
import time
import peft
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# Configure padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
foundation_model.config.pad_token_id = tokenizer.pad_token_id

# Load and sample 10% of the dataset
data = load_dataset("Abirate/english_quotes")["train"].train_test_split(test_size=0.1, seed=42)["test"]
data = data.map(lambda samples: tokenizer(samples["quote"], truncation=True, max_length=128), batched=True)
train_sample = data.select(range(5))
display(train_sample)

# Configure LoRA
lora_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Add the adapter layers to the foundation model
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

output_directory = os.path.join("cache", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    use_cpu=False
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

# --- Clean way to load for inference ---
# Load a completely fresh instance of the base model to avoid the UserWarning
clean_foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
clean_foundation_model.config.pad_token_id = tokenizer.pad_token_id

inference_model = PeftModel.from_pretrained(
    clean_foundation_model,
    peft_model_path,
    is_trainable=False
)

# Generate output tokens
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = inference_model.generate(
    **inputs,
    max_new_tokens=20,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 5.1.0
    Uninstalling datasets-5.1.0:
      Successfully uninstalled datasets-5.1.0


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})

trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.0176
None


Step,Training Loss


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

['Two things are infinite: ']
